# 📊 Notebook 03 — Executive KPI Prep & Validation (Pandas)
Mirrors the SQL curated table `dw_monthly_order_metrics` and exports a CSV for Power BI.

**Concepts called out:**
- `pd.read_csv` (loading data)
- `merge` (SQL join equivalent)
- `groupby + agg` (SQL GROUP BY equivalent)
- Derived columns (revenue, month)
- Export with `to_csv`


In [ ]:
import pandas as pd
from pathlib import Path


## 1) Load raw CSVs
**Business context:** Validate KPIs before publishing a dashboard.

In [ ]:
RAW = Path('../data/raw')
orders = pd.read_csv(RAW/'orders.csv', parse_dates=['order_date'])
order_items = pd.read_csv(RAW/'order_items.csv')
orders.head()

## 2) Build order-level revenue
**Concepts:** derived column, groupby aggregation.

In [ ]:
oi = order_items.copy()
oi['revenue'] = oi['quantity'] * oi['item_price']
order_rev = oi.groupby('order_id', as_index=False).agg(
    order_revenue=('revenue','sum'),
    order_items=('quantity','sum')
)
order_rev.head()

## 3) Join to orders + compute monthly KPIs
**Concepts:** merge, boolean filter, groupby.

In [ ]:
orv = orders.merge(order_rev, on='order_id', how='left')
orv['order_month'] = orv['order_date'].dt.to_period('M').astype(str)
recognized = orv[orv['order_status'].isin(['paid','shipped','delivered'])].copy()

monthly = recognized.groupby('order_month').agg(
    orders_recognized=('order_id','nunique'),
    revenue_recognized=('order_revenue','sum')
).reset_index().sort_values('order_month')
monthly['aov'] = monthly['revenue_recognized'] / monthly['orders_recognized']
monthly.head(12)

## 4) Export for Power BI

In [ ]:
OUT = Path('../data/processed')
OUT.mkdir(parents=True, exist_ok=True)
monthly.to_csv(OUT/'dw_monthly_order_metrics.csv', index=False)
print('Saved:', OUT/'dw_monthly_order_metrics.csv')